<a href="https://colab.research.google.com/github/mbenedicto99/Natural-Language-Processing/blob/main/BENE_RETRIEVER-PECE_2026_Pergunta_e_Resposta_(PR_ou_QA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Marcos Lopes - PECE/USP

# Instalação e importação dos módulos

In [1]:
%%bash

pip install haystack-ai
pip install "datasets>=2.6.1"
pip install "sentence-transformers>=3.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 701.4/701.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.2/240.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.3 MB/s eta 0:00:00


In [2]:
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.readers import ExtractiveReader

# Desabilitar os avisos é opcional, mas ajuda a viver melhor.
import warnings
warnings.filterwarnings("ignore")

# Módulo Retriever

In [3]:
# Conteúdos textuais ("conhecimentos" e pergunta)

documentos = [
    Document(content="Raul Seixas foi um compositor brasileiro."),
    Document(content="Raul Seixas foi um compositor e intérprete brasileiro."),
    Document(content="Raul Seixas foi amigo de Marcelo Nova."),
    Document(content="Paulo Coelho foi parceiro de Raul Seixas."),
    Document(content="Raul Seixas dizia ter nascido há dez mil anos."),
    Document(content="Raul Seixas tinha razão."),
    Document(content="Eu sou a a luz das estrelas")
]

pergunta = "Quem foi Raul Seixas?"

In [4]:
# 1. Criação de um banco de dados com a coleção de documentos
# write_documents() registra os documentos no espaço gerado por InMemoryDocumentStore
# e retorna o número de documentos indexados.

colecao = InMemoryDocumentStore(bm25_algorithm='BM25Okapi', bm25_parameters={'k1': 1.2, 'b': 1})
colecao.write_documents(documentos)

7

In [5]:
# 2. Criação do Retriever (para Recuperação de Informação)

retriever = InMemoryBM25Retriever(document_store=colecao)

In [6]:
# 3. Agora, com o método run() a seguir, será criado um dicionário com
# os documentos recuperados e ordenados por semelhança por relação à pergunta.

docs_recuperados = retriever.run(query=pergunta)

In [7]:
docs_recuperados

{'documents': [Document(id=84c0ea23edd4d4950cdc5ee6df2e262757c2065139c6ff9139ec61f347569deb, content: 'Raul Seixas foi um compositor brasileiro.', score: 0.9040281171615111),
  Document(id=9180b363bf9a49b8b070afd5f0434f0e427d1e298f6357e15564dd7cfe2bf8ad, content: 'Raul Seixas foi amigo de Marcelo Nova.', score: 0.8329247821038641),
  Document(id=68c8c88f447bb9b3b4ad5bbda6f6f0566d798327e04314b27e734c017c4b1022, content: 'Paulo Coelho foi parceiro de Raul Seixas.', score: 0.8329247821038641),
  Document(id=4a2d6c8cf7d99a7f7b64022153950dbb42b633bfbbcb2602f73c545114ba3bc9, content: 'Raul Seixas foi um compositor e intérprete brasileiro.', score: 0.7721906834087906),
  Document(id=ed2f3514cafd56eaa1e3eea8bcf8c18411535e541f33cbbdcc208c5a4475c5d0, content: 'Raul Seixas tinha razão.', score: 0.7267677020318029),
  Document(id=8318013831999ebec68e6bcd83be7b3d2375762f405ed559c0dda3187c7a8f6e, content: 'Raul Seixas dizia ter nascido há dez mil anos.', score: 0.479807803283132),
  Document(id=d281

In [8]:
# Para ficar mais claro...

for e, i in enumerate(docs_recuperados['documents']):
    print(f'({e}) Resposta: {i.content} -- Semelhança: {i.score:.2f}. \n')

(0) Resposta: Raul Seixas foi um compositor brasileiro. -- Semelhança: 0.90. 

(1) Resposta: Raul Seixas foi amigo de Marcelo Nova. -- Semelhança: 0.83. 

(2) Resposta: Paulo Coelho foi parceiro de Raul Seixas. -- Semelhança: 0.83. 

(3) Resposta: Raul Seixas foi um compositor e intérprete brasileiro. -- Semelhança: 0.77. 

(4) Resposta: Raul Seixas tinha razão. -- Semelhança: 0.73. 

(5) Resposta: Raul Seixas dizia ter nascido há dez mil anos. -- Semelhança: 0.48. 

(6) Resposta: Eu sou a a luz das estrelas -- Semelhança: 0.00. 



# Módulo Reader

In [9]:
from haystack.components.readers import ExtractiveReader

In [10]:
# Baixando e inicializado o modelo
reader = ExtractiveReader(model="deepset/roberta-base-squad2-distilled")
reader.warm_up()  # Carrega o modelo na memória

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2-distilled
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/295 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [11]:
docs = docs_recuperados['documents']
resultados_do_reader = reader.run(query=pergunta, documents=docs)

In [12]:
resultados_do_reader

{'answers': [ExtractedAnswer(query='Quem foi Raul Seixas?', score=0.7956740856170654, data='Paulo Coelho', document=Document(id=68c8c88f447bb9b3b4ad5bbda6f6f0566d798327e04314b27e734c017c4b1022, content: 'Paulo Coelho foi parceiro de Raul Seixas.', score: 0.8329247821038641), context=None, document_offset=ExtractedAnswer.Span(start=0, end=12), context_offset=None, meta={}),
  ExtractedAnswer(query='Quem foi Raul Seixas?', score=0.7636266350746155, data='Marcelo Nova', document=Document(id=9180b363bf9a49b8b070afd5f0434f0e427d1e298f6357e15564dd7cfe2bf8ad, content: 'Raul Seixas foi amigo de Marcelo Nova.', score: 0.8329247821038641), context=None, document_offset=ExtractedAnswer.Span(start=25, end=37), context_offset=None, meta={}),
  ExtractedAnswer(query='Quem foi Raul Seixas?', score=0.7234072089195251, data='compositor e intérprete brasileiro', document=Document(id=4a2d6c8cf7d99a7f7b64022153950dbb42b633bfbbcb2602f73c545114ba3bc9, content: 'Raul Seixas foi um compositor e intérprete bra

In [13]:
# Mais legíviel...

for e, i in enumerate(resultados_do_reader['answers']):
    print(f'({e}) Resposta: {i.data}; Semelhança: {i.score:.2f}; Posições na string: {i.document_offset}. \n')

(0) Resposta: Paulo Coelho; Semelhança: 0.80; Posições na string: ExtractedAnswer.Span(start=0, end=12). 

(1) Resposta: Marcelo Nova; Semelhança: 0.76; Posições na string: ExtractedAnswer.Span(start=25, end=37). 

(2) Resposta: compositor e intérprete brasileiro; Semelhança: 0.72; Posições na string: ExtractedAnswer.Span(start=19, end=53). 

(3) Resposta: compositor brasileiro; Semelhança: 0.68; Posições na string: ExtractedAnswer.Span(start=19, end=40). 

(4) Resposta: amigo; Semelhança: 0.59; Posições na string: ExtractedAnswer.Span(start=16, end=21). 

(5) Resposta: razão; Semelhança: 0.58; Posições na string: ExtractedAnswer.Span(start=18, end=23). 

(6) Resposta: um; Semelhança: 0.56; Posições na string: ExtractedAnswer.Span(start=16, end=18). 

(7) Resposta: um; Semelhança: 0.53; Posições na string: ExtractedAnswer.Span(start=16, end=18). 

(8) Resposta: Raul Seixas; Semelhança: 0.44; Posições na string: ExtractedAnswer.Span(start=0, end=11). 

(9) Resposta: Raul Seixas dizia te

# Exercício: Implementação de um *bom* Reader

O Reader que acabamos de implementar funciona muito mal. Afinal, quem seria maluco o bastante para responder à pergunta "Quem foi Raul Seixas" com "Paulo Coelho"?? Bem, se você respondeu a essa nova pergunta com "Raul Seixas", acertou! Mas, para a maioria de nós que não somos tão malucos e geniais como ele, essa não seria a resposta esperada.

Note que o problema só aparece no módulo Reader, não no Retriever. Precisamos, então, implementar um *bom* Reader. Vamos executar as seguintes tarefas:

1) Trocar o modelo de língua do Reader. Aqui vão alguns nomes de modelos muito usados em português:

- neuralmind/bert-base-portuguese-cased (BERTimbau)
- pierreguillou/bert-base-cased-squad-v1.1-portuguese (Um BERT ajustado para P&R no SQuAD 1.1 em português)
- lfcc/bert-portuguese-squad2 (Treinado no dataset SQuAD 2.0 em português)

2) Ainda tendo em mente a troca do modelo de língua, vamos usar um modelo multilíngue, o poderoso XLM-RoBERTa (deepset/xlm-roberta-base-squad2).

Com o modelo multilíngue, vamos tentar fazer perguntas nas seguintes línguas e discutir os resultados:

- Em português ("Quem foi Raul Seixas?");
- Em francês ("Qui était Raul Seixas?");
- Em inglês ("Who was Raul Seixas?");
- Em alemão ("Wer war Raul Seixas?");
- Em japonês ("ラウル・セイシャスは誰でしたか？"), que se lê como "Rauru Seishasu wa dare deshita ka?".

Observe que o XLM-RoBERTa detecta automaticamente a língua utilizada. Você não precisa de nenhum ajuste de parâmetros para mudar a pergunta de uma língua para outra.

3. Nas discussões, observe os seguintes aspectos:

a. Ordenação das respostas;

b. Com quantas respostas o Reader encontrou o alvo? hit@1, hit@3, hit@5...

c. A passagem dada como resposta pelo Reader está "limpa" ou contém ruído de partes irrelevantes do texto?

d. O que informa o grau de semelhança? Dica: não é exatamente o mesmo que o grau de semelhança informado pelo Retriever!


# BERTimbau

# BERT P&R Português - SQuAD 1.1

# BERT P&R -- SQuAD 2.0

# RoBERTa Multilíngue

# Multilíngue em francês

# Multilíngue em inglês

# Multilíngue em alemão

# Multilíngue em japonês

# Retriever de vetores densos

In [14]:
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

In [15]:
# Instanciamento do modelo de vetores densos
store = InMemoryDocumentStore()

# MiniLM é uma versão do Sentence-BERT com 384 dimensões (por sentença, não por palavra)
modelo = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

In [16]:
# Vetorização dos documentos

vetorizador = SentenceTransformersDocumentEmbedder(model=modelo)
vetorizador.warm_up()

embeddings_dos_docs = vetorizador.run(documentos)['documents']
store.write_documents(embeddings_dos_docs)

# O valor True do parâmetro scale_score (opcional) escalona os valores de
# semelhança vetorial no intervalo (0:1)
retriever = InMemoryEmbeddingRetriever(document_store=store, scale_score=True)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
# Verificando a vetorização

primeiro_doc = embeddings_dos_docs[0]

print(f'Texto: {primeiro_doc.content}')
print(f'Tipo: {type(primeiro_doc.embedding)}')
print(f'Dimensões do vetor: {len(primeiro_doc.embedding)}')
print(f'Primeiros 5 valores: {primeiro_doc.embedding[:5]}')

Texto: Raul Seixas foi um compositor brasileiro.
Tipo: <class 'list'>
Dimensões do vetor: 384
Primeiros 5 valores: [-0.17499329149723053, -0.026658551767468452, -0.022197209298610687, 0.17721299827098846, -0.15497666597366333]


In [18]:
# A pergunta também tem de ser vetorizada

text_embedder = SentenceTransformersTextEmbedder(model=modelo)
text_embedder.warm_up()

embeddings_da_pergunta = text_embedder.run(pergunta)['embedding']

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
# Execução

docs_recuperados = retriever.run(query_embedding=embeddings_da_pergunta)

for doc in docs_recuperados['documents']:
    print(f'Conteúdo: {doc.content}; Semelhança: {doc.score:.2f}')

Conteúdo: Raul Seixas foi um compositor brasileiro.; Semelhança: 0.54
Conteúdo: Raul Seixas foi amigo de Marcelo Nova.; Semelhança: 0.54
Conteúdo: Raul Seixas foi um compositor e intérprete brasileiro.; Semelhança: 0.53
Conteúdo: Paulo Coelho foi parceiro de Raul Seixas.; Semelhança: 0.53
Conteúdo: Raul Seixas tinha razão.; Semelhança: 0.53
Conteúdo: Raul Seixas dizia ter nascido há dez mil anos.; Semelhança: 0.53
Conteúdo: Eu sou a a luz das estrelas; Semelhança: 0.51


# Exercício: Perguntas que exploram relações semânticas com vetores densos

Crie duas perguntas novas que explorem relações semânticas não explicitadas no conjunto de documentos. Exemplos dessas relações são:

- Sinônimos (palavras de sentidos semelhantes, mas inéditas no vocabulário);
- Inferências (informações implícitas acarretadas pelas informações explícitas).

Você terá de vetorizar as novas perguntas, mas *não* o conjunto dos documentos.

# Retriever denso + Reader

In [20]:
# Escolha do modelo do Reader

# Alguns parâmetros opcionais:
# no_answer: Não admite respostas vazias
# score_threshold (float): Score mínimo para considerar uma resposta
reader = ExtractiveReader(model="pierreguillou/bert-base-cased-squad-v1.1-portuguese",
                          no_answer=False, score_threshold=0.0)
reader.warm_up()

config.json:   0%|          | 0.00/862 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: pierreguillou/bert-base-cased-squad-v1.1-portuguese
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/494 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [21]:
# Resgata informações do Retriever
documentos_pre_selecionados = docs_recuperados2['documents']

# Executa o Reader
resultados_do_reader = reader.run(
    query=pergunta2,
    documents=documentos_pre_selecionados,
    top_k=11  # Número de respostas
)

print(f'========= A pergunta era: {pergunta2} =========\n')
for e, i in enumerate(resultados_do_reader['answers']):
    print(f'({e+1}) Resposta: {i.data}; Semelhança: {i.score:.2f}; Documento-fonte: <{i.document.content}> \n')

NameError: name 'docs_recuperados2' is not defined

In [ ]:
# Resgata informações do Retriever
documentos_pre_selecionados = docs_recuperados3['documents']

# Executa o Reader
resultados_do_reader = reader.run(
    query=pergunta3,
    documents=documentos_pre_selecionados,
    top_k=11  # Número de respostas
)

print(f'========= A pergunta era: {pergunta3} =========\n')
for e, i in enumerate(resultados_do_reader['answers']):
    print(f'({e+1}) Resposta: {i.data}; Semelhança: {i.score:.2f}; Documento-fonte: <{i.document.content}> \n')